# Week 2: Feature Engineering & Data Preprocessing for Machine Learning

**AnalystLab Africa · Data Science Internship** · **Author:** Molemi Molokoane

**Dataset:** [Heart Failure Prediction](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) (fedesoriano, Kaggle, ODbL licence). 918 patients pooled from five hospital datasets (Cleveland, Hungary, Switzerland, Long Beach VA, Stalog).

**Goal:** turn the raw patient records into a clean, encoded, scaled, machine-learning-ready dataset for predicting whether a patient has heart disease, and document every preprocessing decision.

### Workflow
1. Business understanding
2. Data inspection
3. Exploratory visualisation
4. Data cleaning
5. Feature engineering
6. Train/test split
7. Outlier detection and treatment
8. Feature encoding
9. Feature scaling
10. Feature selection
11. Final machine-learning dataset
12. Answers to the business questions

## 0. Setup

The notebook runs both in Google Colab and locally. If `Data/heart.csv` is not next to the notebook (for example in a fresh Colab session), the raw file is read from the project's GitHub repository.

In [ ]:
# Core libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing and feature-selection tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

# Display and plotting defaults
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42  # fixed seed so every run gives the same split and importances

# Paths: outputs go in folders next to the notebook
DATA_DIR, VIZ_DIR = "Data", "Visualizations"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(VIZ_DIR, exist_ok=True)

LOCAL_CSV = os.path.join(DATA_DIR, "heart.csv")
GITHUB_CSV = ("https://raw.githubusercontent.com/Mo-lemi/AnalystLab-Africa-Data-Science/main/"
              "Week2_DataScience_Project/Data/heart.csv")


def save_fig(name):
    """Save the current figure to the Visualizations folder and show it."""
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, f"{name}.png"), bbox_inches="tight")
    plt.show()

## 1. Business Understanding

Cardiovascular disease is the world's leading cause of death, responsible for roughly 17.9 million deaths a year (WHO). Many of these deaths are preventable if high-risk patients are identified early and referred for further tests or treatment.

- **Business problem:** clinics need a fast, low-cost way to flag patients who are likely to have heart disease, using measurements already taken at a routine visit.
- **Objective:** prepare a dataset from which a classification model can predict heart disease reliably.
- **Target variable:** `HeartDisease` (1 = heart disease, 0 = normal).
- **Why predictive modelling:** a model can screen every patient consistently and instantly, helping clinicians decide who to prioritise for further tests.
- **Expected impact:** earlier diagnosis, better use of scarce specialist time and equipment, and fewer emergency admissions.

The full discussion is in `Reports/Business_Understanding_Report`.

## 2. Data Inspection

### 2.1 Load the dataset and display the first 10 rows

In [ ]:
# Read the raw CSV, locally if available, otherwise from GitHub
source = LOCAL_CSV if os.path.exists(LOCAL_CSV) else GITHUB_CSV
df_raw = pd.read_csv(source)
print(f"Loaded from: {source}")
df_raw.head(10)

### 2.2 Structure: rows, columns and data types

In [ ]:
print(f"Rows: {df_raw.shape[0]}   Columns: {df_raw.shape[1]}\n")
df_raw.info()

In [ ]:
# Group the columns by kind to plan the preprocessing
numeric_cols = df_raw.select_dtypes(include="number").columns.drop("HeartDisease").tolist()
categorical_cols = df_raw.select_dtypes(include="object").columns.tolist()
print("Numeric columns    :", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Target             : HeartDisease")
print("\nUnique values per column:")
df_raw.nunique()

### 2.3 Missing values

`isnull()` finds no gaps, but a value of **0** is impossible for `Cholesterol` or `RestingBP` in a living patient. These zeros are missing values in disguise, so they are counted separately.

In [ ]:
# Standard NaN check
missing = pd.DataFrame({
    "NaN count": df_raw.isnull().sum(),
    "Zero count": (df_raw == 0).sum(),
})
missing["Zero %"] = (missing["Zero count"] / len(df_raw) * 100).round(1)
missing

In [ ]:
# Zeros only mean "missing" for physiological measurements that cannot be 0
hidden_missing = {col: int((df_raw[col] == 0).sum()) for col in ["RestingBP", "Cholesterol"]}
print("Hidden missing values (recorded as 0):", hidden_missing)

# Does missing cholesterol relate to the target?
chol_zero = df_raw["Cholesterol"] == 0
print(f"\nHeart-disease rate when Cholesterol = 0 : {df_raw.loc[chol_zero, 'HeartDisease'].mean():.1%}")
print(f"Heart-disease rate when Cholesterol > 0 : {df_raw.loc[~chol_zero, 'HeartDisease'].mean():.1%}")

**Observation:** 172 patients (18.7%) have `Cholesterol = 0` and one patient has `RestingBP = 0`. Patients with a missing cholesterol reading have a much higher heart-disease rate, so the *fact that it is missing* carries information. These rows come mainly from the Switzerland hospital source, which did not record cholesterol. This is why section 4 keeps a missing-value flag rather than only filling in the gaps.

The 0 values in `FastingBS` (binary flag) and `Oldpeak` (ST depression of 0 mm is normal) are valid.

### 2.4 Duplicate records

In [ ]:
print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Duplicates ignoring the target:", df_raw.drop(columns="HeartDisease").duplicated().sum())

### 2.5 Summary statistics

In [ ]:
df_raw.describe().T.round(2)

In [ ]:
df_raw.describe(include="object").T

In [ ]:
# Class balance of the target
target_counts = df_raw["HeartDisease"].value_counts().rename({0: "Normal (0)", 1: "Heart disease (1)"})
print(target_counts)
print(f"\nPositive class share: {df_raw['HeartDisease'].mean():.1%}")

**Inspection summary**

| Check | Result |
|---|---|
| Shape | 918 rows × 12 columns |
| Data types | 5 numeric (int/float), 5 text categorical, 1 binary flag (`FastingBS`) stored as int, binary target |
| Missing values | No NaNs, but 172 hidden zeros in `Cholesterol` and 1 in `RestingBP` |
| Duplicates | None |
| Target balance | 55% heart disease / 45% normal: mildly imbalanced, so stratified splitting is enough |
| Value ranges | `Oldpeak` has 13 negative values (ST elevation), which are clinically valid; `Cholesterol` reaches 603 mg/dl |

## 3. Exploratory Visualisation

These charts show how each feature relates to the target before any preprocessing. Throughout the notebook, **blue = Normal (0)** and **orange = Heart disease (1)**.

In [ ]:
# Colours chosen to be distinguishable for colour-blind readers
TARGET_COLORS = {0: "#2a78d6", 1: "#eb6834"}
TARGET_LABELS = {0: "Normal", 1: "Heart disease"}

# Figure 1: target distribution
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df_raw, x="HeartDisease", hue="HeartDisease", palette=TARGET_COLORS, legend=False, ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
ax.set_xticks([0, 1], [TARGET_LABELS[0], TARGET_LABELS[1]])
ax.set_title("Figure 1: Target distribution (918 patients)")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Number of patients")
save_fig("01_target_distribution")

In [ ]:
# Figure 2: distribution of each numeric feature, split by diagnosis
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(data=df_raw, x=col, hue="HeartDisease", palette=TARGET_COLORS,
                 bins=30, element="step", ax=ax)
    ax.set_title(f"{col} by diagnosis")
    ax.set_xlabel(col)
    ax.set_ylabel("Number of patients")
    ax.get_legend().set_title("Diagnosis")
    for text, label in zip(ax.get_legend().texts, ["Normal", "Heart disease"]):
        text.set_text(label)
axes.flat[-1].axis("off")  # five numeric features, six panels
fig.suptitle("Figure 2: Numeric feature distributions by diagnosis", fontsize=14)
save_fig("02_numeric_histograms")

**Observations**
- `Cholesterol` has a tall spike at 0. These are the 172 hidden missing values, and almost all of them are heart-disease patients.
- Heart-disease patients are older on average, reach a lower `MaxHR`, and have higher `Oldpeak` (ST depression).
- `Oldpeak` and `Cholesterol` are right-skewed; `RestingBP` has a single 0 value far from the rest.

In [ ]:
# Figure 3: heart-disease counts within each categorical feature
cat_features = ["Sex", "ChestPainType", "FastingBS", "RestingECG", "ExerciseAngina", "ST_Slope"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, cat_features):
    sns.countplot(data=df_raw, x=col, hue="HeartDisease", palette=TARGET_COLORS, ax=ax)
    ax.set_title(f"{col} by diagnosis")
    ax.set_xlabel(col)
    ax.set_ylabel("Number of patients")
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles, ["Normal", "Heart disease"], title="Diagnosis")
fig.suptitle("Figure 3: Categorical features by diagnosis", fontsize=14)
save_fig("03_categorical_countplots")

In [ ]:
# Heart-disease rate in each category, to quantify Figure 3
for col in cat_features:
    rates = df_raw.groupby(col)["HeartDisease"].agg(rate="mean", patients="size")
    rates["rate"] = (rates["rate"] * 100).round(1).astype(str) + "%"
    print(f"\n{col}\n{rates}")

**Observations**
- `ST_Slope`: a *Flat* slope means 83% heart disease vs 20% for *Up*, the strongest single separator.
- `ChestPainType`: *ASY* (asymptomatic) patients have the highest rate (79%). Silent chest pain is a known warning sign.
- `ExerciseAngina = Y` (85%), `FastingBS = 1` (79%) and male sex (63% vs 26%) all raise the risk.
- `RestingECG` separates the classes only weakly (52% to 66%).

In [ ]:
# Figure 4: age against maximum heart rate
fig, ax = plt.subplots(figsize=(8, 5.5))
for label in [0, 1]:
    subset = df_raw[df_raw["HeartDisease"] == label]
    ax.scatter(subset["Age"], subset["MaxHR"], s=28, alpha=0.65, color=TARGET_COLORS[label],
               edgecolor="white", linewidth=0.6, label=TARGET_LABELS[label])
ages = np.arange(df_raw["Age"].min(), df_raw["Age"].max() + 1)
ax.plot(ages, 220 - ages, color="#52514e", linestyle="--", linewidth=1.5, label="Predicted max (220 - age)")
ax.set_title("Figure 4: Age vs maximum heart rate achieved")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Maximum heart rate (bpm)")
ax.legend(title="Diagnosis")
save_fig("04_age_vs_maxhr_scatter")

**Observation:** maximum heart rate falls with age, and heart-disease patients sit lower at every age. Most patients fall well below the age-predicted maximum (220 − age). This motivates an engineered feature: *the percentage of the predicted maximum that the patient reached* (section 5).

## 4. Data Cleaning

### 4.1 Rename columns for clarity
The original names mix abbreviations and styles (`Oldpeak`, `FastingBS`, `ST_Slope`). Descriptive snake_case names make the code and the final dataset easier to read.

In [ ]:
df = df_raw.copy()  # keep the raw data untouched for comparison
column_names = {
    "Age": "age",
    "Sex": "sex",
    "ChestPainType": "chest_pain_type",
    "RestingBP": "resting_bp",
    "Cholesterol": "cholesterol",
    "FastingBS": "fasting_blood_sugar",
    "RestingECG": "resting_ecg",
    "MaxHR": "max_heart_rate",
    "ExerciseAngina": "exercise_angina",
    "Oldpeak": "st_depression",
    "ST_Slope": "st_slope",
    "HeartDisease": "heart_disease",
}
df = df.rename(columns=column_names)
pd.DataFrame({"original": column_names.keys(), "renamed": column_names.values()})

### 4.2 Convert hidden missing values to NaN
A resting blood pressure or cholesterol of 0 is physically impossible, so these zeros are replaced with `NaN`. They will be **imputed after the train/test split** (section 6.2), using training-set medians only, so that no information from the test set leaks into the training data.

Because missing cholesterol is strongly linked to heart disease (section 2.3), a flag column `cholesterol_missing` records which values were missing before the zeros are replaced.

In [ ]:
# Flag first, then replace, so the information is not lost
df["cholesterol_missing"] = (df["cholesterol"] == 0).astype(int)
df[["resting_bp", "cholesterol"]] = df[["resting_bp", "cholesterol"]].replace(0, np.nan)

print("Missing values after conversion:")
print(df.isnull().sum()[df.isnull().sum() > 0])

### 4.3 Remove duplicates

In [ ]:
rows_before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {rows_before - len(df)}  (rows remaining: {len(df)})")

No duplicate patients were found, so no rows were removed. The check stays in the pipeline so the notebook is safe to rerun on updated data.

### 4.4 Validate and correct data types
- Text columns become `category`, which documents that they are labels and uses less memory.
- `fasting_blood_sugar`, `cholesterol_missing` and the target are binary 0/1 integers.
- `resting_bp` and `cholesterol` became floats because they now contain `NaN`; they return to whole numbers after imputation.
- Value ranges are checked against clinically plausible limits.

In [ ]:
categorical_features = ["sex", "chest_pain_type", "resting_ecg", "exercise_angina", "st_slope"]
df[categorical_features] = df[categorical_features].astype("category")
df[["fasting_blood_sugar", "cholesterol_missing", "heart_disease"]] = (
    df[["fasting_blood_sugar", "cholesterol_missing", "heart_disease"]].astype(int))
df.dtypes

In [ ]:
# Range checks: every value should be clinically plausible
checks = {
    "age between 18 and 100": df["age"].between(18, 100).all(),
    "resting_bp between 60 and 250 (ignoring NaN)": df["resting_bp"].dropna().between(60, 250).all(),
    "cholesterol between 80 and 700 (ignoring NaN)": df["cholesterol"].dropna().between(80, 700).all(),
    "max_heart_rate between 50 and 220": df["max_heart_rate"].between(50, 220).all(),
    "st_depression between -3 and 7": df["st_depression"].between(-3, 7).all(),
    "binary columns contain only 0/1": df[["fasting_blood_sugar", "heart_disease"]].isin([0, 1]).all().all(),
}
pd.Series(checks, name="passes")

All ranges pass. The 13 negative `st_depression` values are kept: a negative value means ST *elevation*, which is a real clinical finding, not an error.

## 5. Feature Engineering

New features are added where they carry clinical meaning that the raw columns express only indirectly:

| New feature | Definition | Why |
|---|---|---|
| `cholesterol_missing` | 1 if cholesterol was not recorded | Missingness is linked to heart disease (created in 4.2) |
| `hr_pct_of_predicted` | `max_heart_rate / (220 − age)` | Max heart rate depends on age; this compares each patient with what is expected for their age. Failing to reach it (chronotropic incompetence) is a known cardiac warning sign |
| `has_st_depression` | 1 if `st_depression > 0` | 40% of patients have exactly 0, so whether any depression is present is a meaningful yes/no signal on its own |
| `age_group` | `<45`, `45–54`, `55–64`, `65+` | Clinically used age bands; tested for redundancy with `age` in section 10 |

**Irrelevant features:** the dataset has no ID, name or date columns, so nothing is removed on the grounds of being an identifier. Whether any medical feature should be dropped is decided with evidence in section 10 (feature selection).

In [ ]:
df["hr_pct_of_predicted"] = (df["max_heart_rate"] / (220 - df["age"])).round(3)
df["has_st_depression"] = (df["st_depression"] > 0).astype(int)
df["age_group"] = pd.cut(df["age"], bins=[0, 44, 54, 64, 120],
                         labels=["<45", "45-54", "55-64", "65+"], ordered=True)

df[["age", "max_heart_rate", "hr_pct_of_predicted", "st_depression", "has_st_depression", "age_group"]].head()

In [ ]:
# How do the new features relate to the target?
print("Heart-disease rate by age group:")
print(df.groupby("age_group", observed=True)["heart_disease"].mean().round(3))
print("\nHeart-disease rate by has_st_depression:")
print(df.groupby("has_st_depression")["heart_disease"].mean().round(3))
print("\nMean hr_pct_of_predicted by diagnosis:")
print(df.groupby("heart_disease")["hr_pct_of_predicted"].mean().round(3))

In [ ]:
# Figure 5: the engineered features against the target
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.boxplot(data=df, x="heart_disease", y="hr_pct_of_predicted", hue="heart_disease",
            palette=TARGET_COLORS, legend=False, ax=axes[0])
axes[0].set_xticks([0, 1], ["Normal", "Heart disease"])
axes[0].set_title("% of age-predicted max heart rate")
axes[0].set_xlabel("Diagnosis")
axes[0].set_ylabel("Max HR / (220 − age)")

rate_by_age = df.groupby("age_group", observed=True)["heart_disease"].mean().mul(100)
axes[1].bar(rate_by_age.index.astype(str), rate_by_age.values, color=TARGET_COLORS[1], width=0.6)
axes[1].bar_label(axes[1].containers[0], fmt="%.0f%%", padding=3)
axes[1].set_title("Heart-disease rate by age group")
axes[1].set_xlabel("Age group (years)")
axes[1].set_ylabel("Patients with heart disease (%)")
axes[1].set_ylim(0, 100)

rate_by_missing = df.groupby("cholesterol_missing")["heart_disease"].mean().mul(100)
axes[2].bar(["Recorded", "Missing"], rate_by_missing.values, color=TARGET_COLORS[1], width=0.6)
axes[2].bar_label(axes[2].containers[0], fmt="%.0f%%", padding=3)
axes[2].set_title("Heart-disease rate by cholesterol status")
axes[2].set_xlabel("Cholesterol value")
axes[2].set_ylabel("Patients with heart disease (%)")
axes[2].set_ylim(0, 100)

fig.suptitle("Figure 5: Engineered features vs diagnosis", fontsize=14)
save_fig("05_engineered_features")